In [1]:
!pip install lightgbm


In [2]:
# Cell 1
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix
)

from lightgbm import LGBMClassifier
from scipy.stats import randint, uniform
import joblib


In [3]:

data = pd.read_csv("preprocessed_earthquake_data.csv")

y = data["Status_Reviewed"]

X = data.drop("Status_Reviewed", axis=1)

cat_cols = X.select_dtypes(include=["object"]).columns
print("Categorical columns:", cat_cols.tolist())


X = pd.get_dummies(X, columns=cat_cols, drop_first=True)


from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Categorical columns: ['Type', 'Magnitude Type', 'Source', 'Status']
Train shape: (18727, 60)
Test shape: (4682, 60)


In [4]:

from lightgbm import LGBMClassifier
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV


X_train = X_train.loc[:, ~X_train.columns.duplicated()]
X_test = X_test.loc[:, ~X_test.columns.duplicated()]


lgb_param_dist = {
    "num_leaves": randint(20, 150),
    "max_depth": randint(3, 15),
    "learning_rate": uniform(0.01, 0.2),
    "n_estimators": randint(50, 300),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "reg_alpha": uniform(0, 1),
    "reg_lambda": uniform(0, 1)
}


lgb = LGBMClassifier(random_state=42)


lgb_search = RandomizedSearchCV(
    estimator=lgb,
    param_distributions=lgb_param_dist,
    n_iter=30,
    cv=3,
    scoring="f1",
    verbose=2,
    random_state=42,
    n_jobs=-1,
    error_score="raise"
)


print("Starting LightGBM RandomizedSearchCV ...")
lgb_search.fit(X_train, y_train)


best_lgb = lgb_search.best_estimator_
print("Best LightGBM params:", lgb_search.best_params_)


Starting LightGBM RandomizedSearchCV ...
Fitting 3 folds for each of 30 candidates, totalling 90 fits
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 16616, number of negative: 2111
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000828 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1117
[LightGBM] [Info] Number of data points in the train set: 18727, number of used features: 26
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.887275 -> initscore=2.063204
[LightGBM] [Info] Start training from score 2.063204
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

In [7]:

joblib.dump(best_lgb, "lightgbm_model.pkl")

y_pred_lgb = best_lgb.predict(X_test)
y_proba_lgb = best_lgb.predict_proba(X_test)[:, 1]

print("LightGBM Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_lgb))
print("Precision:", precision_score(y_test, y_pred_lgb))
print("Recall:", recall_score(y_test, y_pred_lgb))
print("F1 Score:", f1_score(y_test, y_pred_lgb))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_lgb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_lgb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lgb))


LightGBM Performance:
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
ROC-AUC: 1.0

Classification Report:
               precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       528
         1.0       1.00      1.00      1.00      4154

    accuracy                           1.00      4682
   macro avg       1.00      1.00      1.00      4682
weighted avg       1.00      1.00      1.00      4682

Confusion Matrix:
 [[ 528    0]
 [   0 4154]]


In [8]:

svm_param_dist = {
    "C": uniform(0.1, 10),
    "gamma": uniform(0.001, 1),
    "kernel": ["rbf", "poly", "sigmoid"]
}

svm = SVC(probability=True, random_state=42)


svm_search = RandomizedSearchCV(
    estimator=svm,
    param_distributions=svm_param_dist,
    n_iter=20,
    cv=3,
    scoring="f1",
    verbose=2,
    random_state=42,
    n_jobs=-1,
    error_score="raise"
)

print("Starting SVM RandomizedSearchCV ...")
svm_search.fit(X_train, y_train)

best_svm = svm_search.best_estimator_
print("Best SVM params:", svm_search.best_params_)


Starting SVM RandomizedSearchCV ...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best SVM params: {'C': np.float64(7.896910002727692), 'gamma': np.float64(0.597850157946487), 'kernel': 'poly'}


In [14]:

joblib.dump(best_svm, "svm_model.pkl")


y_pred_svm = best_svm.predict(X_test)
y_proba_svm = best_svm.predict_proba(X_test)[:, 1]


print("SVM Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print("Precision:", precision_score(y_test, y_pred_svm))
print("Recall:", recall_score(y_test, y_pred_svm))
print("F1 Score:", f1_score(y_test, y_pred_svm))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_svm))
print("\nClassification Report:\n", classification_report(y_test, y_pred_svm))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))


SVM Performance:
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
ROC-AUC: 1.0

Classification Report:
               precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       528
         1.0       1.00      1.00      1.00      4154

    accuracy                           1.00      4682
   macro avg       1.00      1.00      1.00      4682
weighted avg       1.00      1.00      1.00      4682

Confusion Matrix:
 [[ 528    0]
 [   0 4154]]


In [13]:

results = {
    "LightGBM": {
        "Accuracy": accuracy_score(y_test, y_pred_lgb),
        "Precision": precision_score(y_test, y_pred_lgb),
        "Recall": recall_score(y_test, y_pred_lgb),
        "F1": f1_score(y_test, y_pred_lgb),
        "ROC-AUC": roc_auc_score(y_test, y_proba_lgb)
    },
    "SVM": {
        "Accuracy": accuracy_score(y_test, y_pred_svm),
        "Precision": precision_score(y_test, y_pred_svm),
        "Recall": recall_score(y_test, y_pred_svm),
        "F1": f1_score(y_test, y_pred_svm),
        "ROC-AUC": roc_auc_score(y_test, y_proba_svm)
    }
}

print("Model Comparison:")
print(pd.DataFrame(results).T)


Model Comparison:
          Accuracy  Precision  Recall   F1  ROC-AUC
LightGBM       1.0        1.0     1.0  1.0      1.0
SVM            1.0        1.0     1.0  1.0      1.0
